# 🚗 Laboratorio: Regresión Lineal con el caso MPG

## 📖 La historia
En **1973** y **1979** el mundo vivió dos **crisis del petróleo**: la gasolina se volvió cara y escasa. Estados Unidos obligó a los fabricantes a producir autos más eficientes, y los autos japoneses y europeos, más pequeños y ligeros, ganaron mercado.

Tenemos datos de **398 autos** fabricados entre **1970 y 1982**. Somos analistas de una agencia de energía y nos preguntan:

> ### ❓ ¿Qué características de un auto explican su rendimiento de combustible (millas por galón) y qué tan bien podemos predecirlo?

## 🧭 Lo que vas a aprender
1. Explorar un dataset y detectar relaciones entre variables
2. Ajustar e **interpretar** una regresión lineal **simple** y una **múltiple**
3. Evaluar un modelo con datos que **nunca vio** (R² y RMSE)
4. Detectar **multicolinealidad** con el **VIF** y quitar variables redundantes
5. Seleccionar variables **automáticamente** con **RFECV**

## ⏱️ Agenda (3 horas)
| Parte | Tema | Tiempo |
|---|---|---|
| 0 | Preparación | 10 min |
| 1 | Conocer los datos | 15 min |
| 2 | Explorar visualmente | 20 min |
| 3 | Regresión lineal **simple** | 35 min |
| ☕ | Descanso | 10 min |
| 4 | Regresión lineal **múltiple** | 30 min |
| 5 | Multicolinealidad: **VIF** y p-values | 30 min |
| 6 | Selección automática: **RFECV** | 20 min |
| 7 | Conclusiones | 10 min |

## 📋 Las variables
| Variable | Descripción |
|---|---|
| **`mpg`** | 🎯 **Variable objetivo:** millas por galón (más alto = más eficiente) |
| `cylinders` | Número de cilindros del motor |
| `displacement` | Tamaño del motor (pulgadas cúbicas) |
| `horsepower` | Caballos de fuerza |
| `weight` | Peso del auto (libras) |
| `acceleration` | Segundos para acelerar de 0 a 60 mph |
| `model_year` | Año del modelo (70 = 1970) |
| `origin` | Región de origen: `usa`, `europe`, `japan` |
| `name` | Nombre del auto |

---
# 0️⃣ Preparación

Ejecuta las siguientes celdas **una sola vez**. Contienen funciones que te ahorran código repetitivo para que te concentres en **entender** los resultados.

## 🛠️ Funciones auxiliares de visualización

Ejecuta la siguiente celda **una sola vez** al inicio. Después sólo tienes que **llamar** a la función que necesites:

| Función | ¿Para qué sirve? |
|---|---|
| `plot_distributions(df, columnas)` | Histograma + boxplot de variables numéricas |
| `plot_frequencies(df, columnas, top_n=None)` | Frecuencia de variables categóricas |
| `plot_correlation_matrix(df, columnas)` | Matriz de correlación |
| `plot_pairplot(df, columnas, color=None)` | Dispersión entre todas las variables numéricas |
| `plot_simple_regression(x, y, results)` | Recta ajustada de un modelo OLS con 1 variable |
| `plot_actual_vs_predicted(y_real, y_pred)` | Valores reales vs predichos |
| `plot_residuals(y_real, y_pred)` | Residuales vs predichos |
| `plot_rfecv(rfecv)` | R² según el número de variables seleccionadas por RFECV |

In [42]:
# Funciones auxiliares de visualización
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


def plot_distributions(df, columns, nbins=30):
    """Histograma con boxplot marginal para cada variable numérica."""
    for col in columns:
        fig = px.histogram(
            df,
            x=col,
            nbins=nbins,
            marginal='box',
            opacity=0.7,
            title=f'Distribución de {col}'
        )
        fig.update_layout(bargap=0.2)
        fig.show()


def plot_frequencies(df, columns, top_n=None):
    """Gráfica de barras con la frecuencia de cada categoría (top_n limita a las más comunes)."""
    for col in columns:
        freq = df[col].value_counts()
        if top_n:
            freq = freq.head(top_n)
        freq_df = freq.rename_axis(col).reset_index(name='Frecuencia')

        title = f'Frecuencias de {col}'
        if top_n and df[col].nunique() > top_n:
            title += f' (top {top_n})'

        fig = px.bar(freq_df, x=col, y='Frecuencia', title=title)
        fig.update_layout(xaxis={'categoryorder': 'total descending'})
        fig.show()


def plot_correlation_matrix(df, columns):
    """Mapa de calor con la correlación de Pearson entre las variables numéricas."""
    corr = df[columns].corr().round(2)
    fig = px.imshow(
        corr,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        title='Matriz de Correlación'
    )
    fig.update_layout(width=750, height=650)
    fig.show()


def plot_pairplot(df, columns, color=None):
    """Matriz de dispersión (pairplot) entre las variables numéricas."""
    fig = px.scatter_matrix(
        df,
        dimensions=columns,
        color=color,
        title='Pairplot de Variables Numéricas',
        labels={col: col.capitalize() for col in columns}
    )
    fig.update_layout(width=1200, height=1200, title_font_size=20)
    fig.update_traces(diagonal_visible=True)
    fig.show()


def plot_simple_regression(x, y, results):
    """Dispersión de una variable vs el objetivo con la recta ajustada por un OLS de 1 variable."""
    b0, b1 = results.params.iloc[0], results.params.iloc[1]
    x_name = getattr(x, 'name', None) or 'x'
    y_name = getattr(y, 'name', None) or 'y'
    x_line = np.linspace(np.min(x), np.max(x), 100)

    fig = px.scatter(
        x=np.asarray(x),
        y=np.asarray(y),
        opacity=0.6,
        labels={'x': x_name, 'y': y_name},
        title=f'{y_name} = {b0:.2f} + ({b1:.4f}) · {x_name}',
        template='plotly_white'
    )
    fig.add_trace(go.Scatter(
        x=x_line,
        y=b0 + b1 * x_line,
        mode='lines',
        name='Recta OLS',
        line=dict(color='red', width=3)
    ))
    fig.show()


def plot_actual_vs_predicted(y_true, y_pred, title='Real vs Predicho'):
    """Valores reales vs predichos; un modelo perfecto cae sobre la diagonal roja."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    lo = min(y_true.min(), y_pred.min())
    hi = max(y_true.max(), y_pred.max())

    fig = px.scatter(
        x=y_true,
        y=y_pred,
        opacity=0.5,
        labels={'x': 'Valor real', 'y': 'Valor predicho'},
        title=title,
        template='plotly_white'
    )
    fig.add_shape(
        type='line', x0=lo, y0=lo, x1=hi, y1=hi,
        line=dict(color='red', dash='dash')
    )
    fig.show()


def plot_residuals(y_true, y_pred, title='Residuales vs Predicho'):
    """Residuales vs predichos; buscamos una nube sin patrón alrededor de 0."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)

    fig = px.scatter(
        x=y_pred,
        y=y_true - y_pred,
        opacity=0.5,
        labels={'x': 'Valor predicho', 'y': 'Residual (real − predicho)'},
        title=title,
        template='plotly_white'
    )
    fig.add_hline(y=0, line_dash='dash', line_color='red')
    fig.show()


def plot_rfecv(rfecv):
    """R² promedio de validación cruzada según el número de variables que conserva RFECV."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=rfecv.cv_results_['n_features'],
        y=rfecv.cv_results_['mean_test_score'],
        mode='lines+markers',
        line=dict(color='steelblue', width=3),
        marker=dict(size=7),
        name='R² promedio (CV)'
    ))
    fig.update_layout(
        title='RFECV — R² según número de variables seleccionadas',
        xaxis_title='Número de variables',
        yaxis_title='R² (validación cruzada)',
        template='plotly_white',
        width=900, height=450
    )
    fig.show()

### 🧰 Funciones de modelado

En la Parte 3 escribirás el modelo **a mano** para entender cada paso. Después usaremos estas funciones, que hacen exactamente lo mismo en una línea:

| Función | ¿Para qué sirve? |
|---|---|
| `ajustar_ols(X_train, y_train)` | Ajusta un modelo OLS (ya agrega la constante) |
| `predecir(results, X)` | Genera predicciones con el modelo |
| `evaluar_modelo(nombre, results, X_test, y_test)` | R² y RMSE en el conjunto de prueba |
| `calcular_vif(X)` | VIF de cada variable, de mayor a menor |

In [43]:
# Funciones de modelado: ajustar, evaluar y diagnosticar modelos OLS
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import r2_score, mean_squared_error
from statsmodels.stats.outliers_influence import variance_inflation_factor


def ajustar_ols(X_train, y_train):
    """Ajusta un modelo OLS (agrega la constante automáticamente)."""
    return sm.OLS(y_train, sm.add_constant(X_train)).fit()


def predecir(results, X):
    """Predice con un modelo OLS ajustado con ajustar_ols()."""
    return results.predict(sm.add_constant(X))


def evaluar_modelo(nombre, results, X_test, y_test):
    """Imprime y regresa las métricas del modelo en el conjunto de prueba."""
    y_pred = predecir(results, X_test)
    metricas = {
        'modelo': nombre,
        'n_variables': X_test.shape[1],
        'R² ajustado (train)': round(results.rsquared_adj, 4),
        'R² (test)': round(r2_score(y_test, y_pred), 4),
        'RMSE (test)': round(np.sqrt(mean_squared_error(y_test, y_pred)), 4),
    }
    print(f"{nombre}: R² test = {metricas['R² (test)']} | RMSE test = {metricas['RMSE (test)']}")
    return metricas


def calcular_vif(X):
    """VIF de cada variable, de mayor a menor (se calcula con constante, igual que el modelo)."""
    X_const = sm.add_constant(X)
    vif = pd.DataFrame({
        'variable': X.columns,
        'VIF': [variance_inflation_factor(X_const.values, i + 1) for i in range(X.shape[1])],
    })
    return vif.sort_values('VIF', ascending=False).round(2).reset_index(drop=True)

### 📥 Cargar los datos
Los datos viven en una base de datos **SQLite**. Esta celda la descarga y trae una tabla con una consulta SQL.

In [44]:
import requests, sqlite3, pandas as pd

url = "https://raw.githubusercontent.com/davidjamesknight/SQLite_databases_for_learning_data_science/main/mpg.db"
r = requests.get(url)

with open("mpg.db", "wb") as f:
    f.write(r.content)

conn = sqlite3.connect("mpg.db")

query = """
SELECT
    O.mpg,
    O.cylinders,
    O.displacement,
    O.horsepower,
    O.weight,
    O.acceleration,
    O.model_year,
    ORG.origin,
    N.name
FROM
    Observation AS O
JOIN
    Origin AS ORG ON O.origin_id = ORG.origin_id
JOIN
    Name AS N ON O.name_id = N.name_id
"""

df = pd.read_sql_query(query, conn)
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite
3,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst
4,17.0,8,302.0,140.0,3449,10.5,70,usa,ford torino


---
# 1️⃣ Conocer los datos (15 min)

Antes de modelar respondemos: ¿qué tipo de datos tenemos?, ¿faltan valores?, ¿en qué rangos se mueven?

In [45]:
# ¿Cuántas filas y columnas? ¿Qué tipo de dato tiene cada columna?
# Pista: df.shape y df.dtypes
# ✍️ Tu código aquí
df.shape
df.type

AttributeError: 'DataFrame' object has no attribute 'type'

In [ ]:
# ¿Hay valores nulos?
# Pista: df.isnull().sum()
# ✍️ Tu código aquí
df.isnull().sum()

### Tratamiento de nulos
**Regla práctica:** si los nulos son **menos del 5%** de las filas, se pueden eliminar; si son más, conviene **imputarlos** (rellenarlos, por ejemplo con la mediana).

In [ ]:
# 1. Calcula el porcentaje de nulos en horsepower. Pista: df['horsepower'].isnull().mean()
# 2. Si es menor al 5%, elimina esas filas: df = df.dropna(subset=['horsepower'])
# ✍️ Tu código aquí
df['horsepower'].isnull().mean()
df = df.dropna(subset=['horsepower'])

In [ ]:
# Estadísticas descriptivas
# Pista: df.describe().round(1)
# ✍️ Tu código aquí
df.describe().round(1)

✍️ **¿Qué observas?** ¿Qué columna tenía nulos y cuántos? ¿En qué rango se mueve `mpg`? ¿Y `weight`?

RELLENAR CON TUS COMENTARIOS

---
# 2️⃣ Explorar visualmente (20 min)

Buscamos pistas: ¿qué variables se mueven junto con `mpg`?

In [ ]:
# 1. Crea la lista numerical_vars con las 7 columnas numéricas
# 2. Llama a plot_distributions(df, numerical_vars)
# ✍️ Tu código aquí
numerical_vars=[
    'mpg',
    'cylinders',
    'displacement',
    'horsepower',
    'weight',
    'acceleration',
    'model_year'
]
plot_distributions(df,numerical_vars)

In [ ]:
# ¿Cuántos autos hay de cada origen?
# Llama a plot_frequencies(df, ['origin'])
# ✍️ Tu código aquí

plot_frequencies(df,["name"])

In [ ]:
# Matriz de correlación
# Llama a plot_correlation_matrix(df, numerical_vars)
# ✍️ Tu código aquí
plot_correlation_matrix(df, numerical_vars)

In [ ]:
# Este va de regalo: cada punto es un auto, coloreado por origen
plot_pairplot(df, numerical_vars, color='origin')

✍️ **¿Qué observas?**
- ¿Qué variable está más correlacionada con `mpg`? ¿La relación es positiva o negativa?
- ¿Qué variables están muy correlacionadas **entre sí**?
- ¿Los autos más nuevos rinden más o menos?
-Weight relacion negatica
-Displacement, horsepower, cylinders
-rinden mas
RELLENAR CON TUS COMENTARIOS

---
# 3️⃣ Regresión lineal simple (35 min)

Empezamos con **una sola variable**: la más correlacionada con `mpg`, el **peso**.

$$\text{mpg} = \beta_0 + \beta_1 \cdot \text{weight}$$

### Paso 1: separar variables y dividir en entrenamiento y prueba
- `X` → variables **predictoras** (todo menos `mpg` y `name`)
- `y` → variable **objetivo** (`mpg`)

El modelo **aprende** con el 80% de los autos (**train**) y lo **evaluamos** con el 20% restante (**test**). Es como estudiar con unos ejercicios y hacer el examen con otros: así sabemos si el modelo realmente aprendió o sólo memorizó.

In [ ]:
from sklearn.model_selection import train_test_split

# 1. X = df sin las columnas 'mpg' y 'name'   → df.drop(columns=[...])
# 2. y = la columna 'mpg'
# 3. X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# ✍️ Tu código aquí
X= df.drop(columns=["mpg","name"])
y= df["mpg"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

Train: (318, 7) | Test: (80, 7)


### Paso 2: ajustar el modelo
Usamos `statsmodels`. Hay que **agregar una constante** para que el modelo calcule la ordenada al origen ($\beta_0$).

In [ ]:
import statsmodels.api as sm

# 1. X_train_simple = sm.add_constant(X_train[['weight']])
# 2. modelo_simple = sm.OLS(y_train, X_train_simple).fit()
# 3. print(modelo_simple.summary())
# ✍️ Tu código aquí
X_train_simple = sm.add_constant(X_train[['weight']])
modelo_simple = sm.OLS(y_train, X_train_simple).fit()
print(modelo_simple.summary())

                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       0.684
Model:                            OLS   Adj. R-squared:                  0.683
Method:                 Least Squares   F-statistic:                     685.5
Date:                Fri, 25 Sep 2026   Prob (F-statistic):           3.80e-81
Time:                        17:30:22   Log-Likelihood:                -925.80
No. Observations:                 318   AIC:                             1856.
Df Residuals:                     316   BIC:                             1863.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         46.7821      0.920     50.862      0.0

### 🔍 Cómo leer el resumen: sólo 3 cosas
El resumen tiene muchos números. Por ahora fíjate en:

| Dónde | Qué es |
|---|---|
| `R-squared` (arriba a la derecha) | % de la variación de `mpg` que explica el modelo |
| Columna `coef` | Los coeficientes $\beta_0$ (`const`) y $\beta_1$ (`weight`) |
| Columna `P>\|t\|` | **p-value**: si es < 0.05, la variable es **significativa** |

✍️ Escribe los 3 valores de tu modelo:
- R² = .698
- $\beta_0$ = , $\beta_1$ = 47.2, -.0079

- p-value de `weight` = 0

In [ ]:
# Visualizar la recta ajustada
# Llama a plot_simple_regression(X_train['weight'], y_train, modelo_simple)
# ✍️ Tu código aquí
plot_simple_regression(X_train['weight'], y_train, modelo_simple)

### 💬 Interpretación en palabras
✍️ Completa: *"Por cada 1,000 libras adicionales de peso, el rendimiento del auto disminuye en 7.9 mpg."*

✍️ Usa la ecuación para estimar el rendimiento de un auto de 2,000 lb y uno de 4,000 lb.

RELLENAR CON TUS COMENTARIOS
- 47.20 - 0.0079 * 2000 = 31.4
- 47.20 - 0.0079 * 4000 = 15.6


### Paso 3: evaluar con datos de prueba
- **R²**: qué tanto explica el modelo (0 = nada, 1 = perfecto)
- **RMSE**: error típico **en las mismas unidades que `mpg`**

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# 1. Agrega la constante a X_test[['weight']]      → X_test_simple
# 2. Predice: y_pred_simple = modelo_simple.predict(X_test_simple)
# 3. r2 = r2_score(y_test, y_pred_simple)
# 4. rmse = np.sqrt(mean_squared_error(y_test, y_pred_simple))
# 5. Imprime ambos
# ✍️ Tu código aquí
X_test_simple = sm.add_constant(X_test[['weight']])
y_pred_simple= modelo_simple.predict(X_test_simple)

r2= r2_score(y_test,y_pred_simple)
rmse= np.sqrt(mean_squared_error(y_test,y_pred_simple))
print(f"R2 en test: {r2:.3}")
print(f"Rmse en test:{rmse:.2f}mpg")

R2 en test: 0.723
Rmse en test:3.86mpg


In [ ]:
# Llama a plot_actual_vs_predicted(y_test, y_pred_simple) y a plot_residuals(y_test, y_pred_simple)
# ✍️ Tu código aquí
plot_actual_vs_predicted(
    y_test,
    y_pred_simple,
    title="Regresion simple - Real vs predicho")

plot_residuals(y_test, y_pred_simple)

✍️ ¿Qué R² y RMSE obtuviste en test? Explica en palabras qué significa el RMSE.

R2 en test: 0.723
Rmse en test:3.86mpg


### 📊 Guardamos los resultados
Vamos a comparar todos los modelos al final. A partir de aquí usamos `evaluar_modelo()`, que calcula exactamente lo mismo que acabas de escribir a mano.

In [ ]:
comparacion = []
comparacion.append(evaluar_modelo('1. Simple (weight)', modelo_simple, X_test[['weight']], y_test))

1. Simple (weight): R² test = 0.723 | RMSE test = 3.8594


---
# ☕ Descanso (10 min)

---
# 4️⃣ Regresión lineal múltiple (30 min)

Ahora usamos **todas** las variables:

$$\text{mpg} = \beta_0 + \beta_1 \cdot \text{cylinders} + \beta_2 \cdot \text{displacement} + \dots + \beta_k \cdot \text{origin}$$

### Paso 1: convertir `origin` en números (One-Hot Encoding)
Un modelo sólo entiende números. Convertimos `origin` en columnas de **0 y 1** (variables *dummy*).

⚠️ **Necesitamos una columna menos que categorías.** Si un auto **no** es europeo **ni** japonés, ya sabemos que es americano: una tercera columna sería redundante y el modelo no podría calcular sus coeficientes (*trampa de las variables dummy*).

Quitamos la columna de `usa`, que será la **categoría de referencia**: los coeficientes de `origin_europe` y `origin_japan` se leen **comparados con los autos americanos**.

In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(drop=['usa'], sparse_output=False)

# fit_transform sólo en train; en test sólo transform (con lo aprendido en train)
origin_train = pd.DataFrame(
    ohe.fit_transform(X_train[['origin']]),
    columns=ohe.get_feature_names_out(),
    index=X_train.index
)
origin_test = pd.DataFrame(
    ohe.transform(X_test[['origin']]),
    columns=ohe.get_feature_names_out(),
    index=X_test.index
)

# Reemplazamos la columna 'origin' por sus columnas dummy
X_train_enc = pd.concat([X_train.drop(columns=['origin']), origin_train], axis=1)
X_test_enc = pd.concat([X_test.drop(columns=['origin']), origin_test], axis=1)

X_train_enc.head()

,cylinders,displacement,horsepower,weight,acceleration,model_year,origin_europe,origin_japan
3,8,304.0,150.0,3433,12.0,70,0.0,0.0
18,4,97.0,88.0,2130,14.5,70,0.0,1.0
376,4,91.0,68.0,2025,18.2,82,0.0,1.0
248,4,91.0,60.0,1800,16.4,78,0.0,1.0
177,4,115.0,95.0,2694,15.0,75,1.0,0.0


### Paso 2: ajustar el modelo con todas las variables

In [46]:
# 1. modelo_multiple = ajustar_ols(X_train_enc, y_train)
# 2. print(modelo_multiple.summary())
# ✍️ Tu código aquí
modelo_multiple = ajustar_ols(X_train_enc, y_train)
print(modelo_multiple.summary())

MissingDataError: exog contains inf or nans

In [40]:
# Evalúa el modelo y agrégalo a la comparación
# Pista: comparacion.append(evaluar_modelo('2. Múltiple (todas)', modelo_multiple, X_test_enc, y_test))
# ✍️ Tu código aquí
comparacion.append(
    evaluar_modelo(
        '2. Múltiple (todas)', 
        modelo_multiple, 
        X_test_enc, y_test))

NameError: name 'modelo_multiple' is not defined

### 💬 Interpretación
En regresión múltiple cada coeficiente se lee **"manteniendo las demás variables constantes"**.

✍️ Interpreta en palabras los coeficientes de `weight`, `model_year` y `origin_japan`.

✍️ ¿Mejoró el R² en test respecto a la regresión simple? ¿Y el RMSE?

### 🚩 Busca lo que no cuadra
✍️ Revisa el **signo** del coeficiente de `displacement`. ¿Tiene sentido que un motor más grande haga que el auto rinda más?

✍️ ¿Qué variables tienen p-value > 0.05? ¿No habíamos visto en la Parte 2 que estaban muy relacionadas con `mpg`?

RELLENAR CON TUS COMENTARIOS

---
# 5️⃣ Multicolinealidad: VIF y p-values (30 min)

**Multicolinealidad** = varias predictoras cuentan **la misma historia**. `weight`, `displacement`, `cylinders` y `horsepower` miden, en el fondo, **qué tan grande es el auto**. El modelo no sabe a cuál darle el crédito y sus coeficientes se vuelven **inestables**: signos raros y p-values altos.

El **VIF** (*Variance Inflation Factor*) mide qué tanto se puede explicar una variable **con las demás**:

| VIF | Interpretación |
|---|---|
| < 5 | ✅ Sin problema |
| 5 – 10 | ⚠️ Multicolinealidad moderada |
| > 10 | ❌ Multicolinealidad severa |

**Receta:** quita la variable con el VIF más alto, **vuelve a calcular** (al quitar una, las demás cambian) y repite hasta que todas queden por debajo de 5.

In [ ]:
# Calcula el VIF de todas las variables
# Pista: calcular_vif(X_train_enc)
# ✍️ Tu código aquí
calcular_vif(X_train_enc)

✍️ ¿Qué variable tiene el VIF más alto? Quítala y vuelve a calcular.

In [ ]:
# cols_vif = X_train_enc.columns.drop('<variable con mayor VIF>')
# calcular_vif(X_train_enc[cols_vif])
# ✍️ Tu código aquí
cols_vif = X_train_enc.columns.drop('displacement')
calcular_vif(X_train_enc[cols_vif])

✍️ ¿Cuál es ahora la más alta? Quítala también (partiendo de `cols_vif`).

In [ ]:
# cols_vif = cols_vif.drop('<variable>')
# calcular_vif(X_train_enc[cols_vif])
# ✍️ Tu código aquí
cols_vif = cols_vif.drop('horsepower')
calcular_vif(X_train_enc[cols_vif])

✍️ ¿Todavía hay alguna por encima de 5? Repite.

In [ ]:
# cols_vif = cols_vif.drop('<variable>')
# calcular_vif(X_train_enc[cols_vif])
# ✍️ Tu código aquí
cols_vif = cols_vif.drop('cylinders')
calcular_vif(X_train_enc[cols_vif])

✍️ ¿Qué variable de "tamaño" sobrevivió?

### Ahora los p-values
Ajusta el modelo con las variables que quedaron y revisa si todas son **significativas** (p < 0.05).

In [ ]:
# 1. modelo_vif = ajustar_ols(X_train_enc[cols_vif], y_train)
# 2. Muestra los p-values: modelo_vif.pvalues.round(4)
# ✍️ Tu código aquí
modelo_vif = ajustar_ols(X_train_enc[cols_vif], y_train)
modelo_vif.pvalues.round(4)

✍️ ¿Qué variable tiene p-value > 0.05? Quítala y ajusta el modelo final.

In [ ]:
# 1. cols_vif = cols_vif.drop('<variable>')
# 2. modelo_vif = ajustar_ols(X_train_enc[cols_vif], y_train)
# 3. print(modelo_vif.summary())
# ✍️ Tu código aquí
cols_vif = cols_vif.drop('acceleration')
modelo_vif = ajustar_ols(X_train_enc[cols_vif], y_train)
print(modelo_vif.summary())

In [ ]:
# Evalúa modelo_vif y agrégalo a la comparación (usa X_test_enc[cols_vif])
# ✍️ Tu código aquí
comparacion.append(
    evaluar_modelo(
        "3. VIF+ P-Values",
        modelo_vif,
        X_test_enc[cols_vif],
        y_test
    )
)

✍️ ¿Con cuántas variables terminaste? Compara el R² en test con el del modelo múltiple. ¿Los coeficientes ahora tienen signos lógicos?

RELLENAR CON TUS COMENTARIOS

---
# 6️⃣ Selección automática: RFECV (20 min)

Lo que hicimos a mano lo puede hacer un algoritmo. **RFECV** (*Recursive Feature Elimination with Cross-Validation*):

1. Entrena el modelo con todas las variables
2. Elimina la **menos importante** (la de coeficiente más pequeño)
3. Repite hasta quedarse con 1 variable
4. En cada paso mide el R² con **validación cruzada** (5 particiones distintas de train) y se queda con el mejor número de variables

⚠️ **Hay que escalar las variables primero.** RFECV decide por el **tamaño del coeficiente**, y ese tamaño depende de las unidades: `weight` está en libras, así que su coeficiente es diminuto (-0.006) aunque sea la variable más importante. Con `StandardScaler` todas quedan en la misma escala.

In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
from sklearn.feature_selection import RFECV, RFE
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

# Escalar (fit sólo con train)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_enc),
    columns=X_train_enc.columns,
    index=X_train_enc.index
)

rfecv = RFECV(
    estimator=LinearRegression(),
    step=1,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring='r2'
)
rfecv.fit(X_train_scaled, y_train)

print(f"Número óptimo de variables: {rfecv.n_features_}")

In [ ]:
# Grafica el R² según el número de variables
# Llama a plot_rfecv(rfecv)
# ✍️ Tu código aquí


✍️ ¿Cuántas variables eligió RFECV? Observa la curva: ¿a partir de cuántas variables casi deja de mejorar el R²?

Cuando la curva se **aplana**, preferimos el modelo **más simple** (*principio de parsimonia*). Usamos `RFE` para pedirle directamente **las 2 mejores variables**:

In [ ]:
# 1. rfe = RFE(estimator=LinearRegression(), n_features_to_select=2)
# 2. rfe.fit(X_train_scaled, y_train)
# 3. cols_rfe = X_train_enc.columns[rfe.support_]
# 4. Imprime cols_rfe
# ✍️ Tu código aquí


In [ ]:
# Ajusta modelo_rfe con cols_rfe (sin escalar: X_train_enc[cols_rfe]), muestra el summary,
# evalúalo y agrégalo a la comparación
# ✍️ Tu código aquí


✍️ ¿Qué 2 variables eligió RFE? ¿Coinciden con las que sobrevivieron al VIF? ¿Cómo es su R² en test comparado con el modelo de 8 variables?

RELLENAR CON TUS COMENTARIOS

---
# 7️⃣ Conclusiones (10 min)

In [ ]:
# Tabla comparativa de todos los modelos
# Pista: pd.DataFrame(comparacion)
# ✍️ Tu código aquí


In [ ]:
# Grafica real vs predicho del modelo final
# 1. y_pred_rfe = predecir(modelo_rfe, X_test_enc[cols_rfe])
# 2. plot_actual_vs_predicted(y_test, y_pred_rfe)
# ✍️ Tu código aquí


## ❓ Respuesta a la pregunta
> *¿Qué características explican el rendimiento de un auto y qué tan bien podemos predecirlo?*

✍️ Responde con tus resultados: ¿qué variables importan más? ¿Qué tan preciso es tu mejor modelo?

RELLENAR CON TUS COMENTARIOS

## 📝 Lo que aprendimos
✍️ Escribe 3 cosas que aprendiste hoy.

💡 En Machine Learning puro se prioriza el R². En econometría y ciencias sociales se prioriza la validez estadística. Ambos mundos son válidos, pero tienen distintos criterios de éxito.

## 🚀 Retos opcionales
- Repite la regresión simple usando `horsepower` en lugar de `weight`. ¿Cuál predice mejor?
- Usa `RFE` con `n_features_to_select=4`. ¿Qué variables elige? Compáralas con las que quedaron después del VIF y los p-values